Реализуйте телеграм-бота, который будет использовать предварительно обученную модель и классифицировать изображение, присланное пользователем.

In [4]:
import torch
from PIL import Image
from io import BytesIO

from torchvision.models import resnet18, ResNet18_Weights

from telegram import Update
from telegram.ext import (
    Application,
    CommandHandler,
    MessageHandler,
    ContextTypes,
    filters
)

In [8]:
# 1. Выбираем устройство
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# 2. Загружаем предварительно обученную модель
weights = ResNet18_Weights.DEFAULT

model = resnet18(weights=weights)

model = model.to(device)
model.eval()


# 3. Получаем преобразования изображений
preprocess = weights.transforms()

# Названия 1000 классов ImageNet
categories = weights.meta["categories"]


# 4. Обработчик команды /start
async def start(
    update: Update,
    context: ContextTypes.DEFAULT_TYPE
):
    await update.message.reply_text(
        "Привет! Отправь мне фотографию, "
        "и я попробую определить, что на ней изображено."
    )


# 5. Функция классификации изображения
def predict_image(image):

    # Преобразуем изображение в тензор
    img_tensor = preprocess(image)

    # Добавляем размерность batch
    img_tensor = img_tensor.unsqueeze(0)

    # Переносим на устройство
    img_tensor = img_tensor.to(device)

    # Получаем предсказание модели
    with torch.inference_mode():
        outputs = model(img_tensor)

        probabilities = torch.softmax(outputs, dim=1)

        # Три наиболее вероятных класса
        top_prob, top_indices = torch.topk(
            probabilities,
            k=3,
            dim=1
        )

    results = []

    for prob, idx in zip(
        top_prob[0],
        top_indices[0]
    ):
        class_name = categories[idx.item()]
        confidence = prob.item() * 100

        results.append(
            f"{class_name}: {confidence:.2f}%"
        )

    return results


# 6. Обработчик фотографий
async def handle_photo(
    update: Update,
    context: ContextTypes.DEFAULT_TYPE
):

    await update.message.reply_text(
        "Получила изображение! Анализирую..."
    )

    # Получаем фотографию максимального размера
    photo = update.message.photo[-1]

    # Получаем файл из Telegram
    telegram_file = await context.bot.get_file(
        photo.file_id
    )

    # Загружаем изображение в память
    image_bytes = BytesIO()

    await telegram_file.download_to_memory(
        out=image_bytes
    )

    image_bytes.seek(0)

    # Открываем изображение
    image = Image.open(image_bytes).convert("RGB")

    # Классифицируем изображение
    predictions = predict_image(image)

    # Формируем ответ
    answer = "Результат классификации:\n\n"

    for prediction in predictions:
        answer += prediction + "\n"

    # Отправляем результат пользователю
    await update.message.reply_text(answer)


# 7. Запускаем бота

TOKEN = "8790825617:AAEHRypgLPMe51Ls9p70Z4K-VsCiRF7A4YQ"

application = (
    Application.builder()
    .token(TOKEN)
    .build()
)

application.add_handler(
    CommandHandler("start", start)
)

application.add_handler(
    MessageHandler(filters.PHOTO, handle_photo)
)


# Запускаем бота в Jupyter
await application.initialize()
await application.start()
await application.updater.start_polling()

print("Бот запущен!")

Бот запущен!


In [3]:
%pip install python-telegram-bot

  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached anyio-4.16.0-py3-none-any.whl.metadata (4.7 kB)
  Using cached certifi-2026.10.23-py3-none-any.whl.metadata (2.5 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
   ---------------------------------------- 0.0/769.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/769.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/769.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/769.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/769.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/769.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/769.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/769.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/769.


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
await application.updater.stop()
await application.stop()
await application.shutdown()

print("Бот остановлен!")

Бот остановлен!
